# 01 Data Exploration

This notebook covers EDA for intraday OHLCV, engineered features, class balance, and mutual information ranking.

In [ ]:
# Step 1: load configuration and basic dependencies.
from src.utils.config import load_config, flatten_tickers
from src.data.download import download_ticker_data
from src.data.features import engineer_features
from src.data.labels import build_labels
from src.evaluation.feature_importance import mutual_information_ranking

import matplotlib.pyplot as plt
import seaborn as sns

config = load_config('config/default.yaml')
all_tickers = flatten_tickers(config['tickers'])
all_tickers[:5]

In [ ]:
# Step 2: download one ticker for quick exploratory analysis.
ticker = all_tickers[0]
raw_df = download_ticker_data(ticker=ticker, config=config, force_refresh=False)
raw_df.head()

In [ ]:
# Step 3: engineer all planned features and volatility-normalized labels.
feature_df, feature_cols, _ = engineer_features(raw_df, config=config, fit_normalizer=False)
labeled_df, _ = build_labels(feature_df, config=config)
labeled_df[feature_cols + ['label']].head()

In [ ]:
# Step 4: inspect class balance to verify label thresholds.
class_balance = labeled_df['label'].value_counts(normalize=True).sort_index()
class_balance

In [ ]:
# Step 5: visualize feature correlation structure.
plt.figure(figsize=(12, 8))
corr = labeled_df[feature_cols].corr().iloc[:20, :20]
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.title('Feature Correlation (First 20 Features)')
plt.tight_layout()

In [ ]:
# Step 6: compute mutual information feature ranking.
mi_df = mutual_information_ranking(labeled_df, feature_cols)
mi_df.head(15)